# 差分メモ

- 対象: `[2]dpc-starter-train-v3.ipynb`
- 元: `[2]dpc-starter-train-v2.ipynb`
- 種別: バージョン更新
- 変更点:
  - 学習データを curated 版 `train.curated.v002.xlsx` に差し替え（`oare_id`, `transliteration_x`, `translation` を使用）
  - 前処理/後処理の正規化に `.codex/docs/normalization_replacements.md` の観点（`<gap>` 統一・縮約、下付き数字、長い小数の短縮など）を反映
  - `<gap>. <gap>` / `<gap>, <gap>` のように句読点を挟む連続 `<gap>` も `<gap>` に縮約
- 変更理由/仮説:
  - 目視修正した train（curated）を使い、train.csv 由来のノイズを減らしたい
  - `<gap>` まわりの揺れを縮約して token パターンの一貫性を上げたい



# Deep Past Initiative – Machine Translation (Training Notebook)

This notebook is a **starter / baseline** for this Kaggle competition.

Main ideas:
- Use **ByT5** to handle noisy Akkadian transliterations at the character level
- Perform **simple sentence alignment** to increase training data
- Fine-tune using HuggingFace `Trainer`


Inference Code is [here](https://www.kaggle.com/code/takamichitoda/dpc-starter-infer).

In [1]:
# (Optional) Kaggle environment usually includes required libs.
# If your environment is missing something, uncomment and install as needed.
# !pip install -q datasets transformers accelerate


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import re
import math
import unicodedata
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

2026-03-13 14:59:04.117785: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773413944.309019      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773413944.366464      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773413944.849053      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773413944.849094      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773413944.849097      24 computation_placer.cc:177] computation placer alr

In [3]:
class Config:
    # Akkadian transliteration contains a lot of noise and many unknown words, so
    # ByT5, which processes text at the character (byte) level rather than the word level, is a strong choice.
    MODEL_NAME = "google/byt5-base"

    # ByT5 tends to produce longer token sequences, but 512 tokens is enough at the sentence level.
    MAX_LENGTH = 512

    # Training
    EPOCHS = 20
    LEARNING_RATE = 2e-4

    # ByT5-base is heavier; keep per-device batch small and use grad-accum.
    PER_DEVICE_TRAIN_BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 8

    # Output
    OUTPUT_DIR = "./byt5-akkadian-model/fulltrain_byt5-base_multitask"


In [4]:
# Fix the seed (for reproducibility).
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    
seed_everything()

In [5]:
INPUT_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"

# ------------------------------------------
# Train: curated xlsx を優先して読む
# ------------------------------------------
# Kaggle 上で使う場合は、curated xlsx を Dataset として追加して、
# `/kaggle/input/.../train.curated.v002.xlsx` が参照できるようにしてください。
USE_CURATED_TRAIN = True
CURATED_TRAIN_XLSX_CANDIDATES = [
    "/kaggle/input/datasets/tatsuokoshida/dpc-curated-train/train.curated.v002.xlsx",
]


def load_train_df():
    if USE_CURATED_TRAIN:
        curated_path = None
        for p in CURATED_TRAIN_XLSX_CANDIDATES:
            if os.path.exists(p):
                curated_path = p
                break
        if curated_path is None:
            raise FileNotFoundError(
                "curated train xlsx が見つかりません。CURATED_TRAIN_XLSX_CANDIDATES を確認してください。"
            )

        df = pd.read_excel(curated_path)
        df = df.rename(columns={"transliteration_x": "transliteration"})

        required_cols = {"oare_id", "transliteration", "translation"}
        missing_cols = required_cols - set(df.columns)
        assert not missing_cols, f"curated xlsx の必須カラムが不足しています: {sorted(missing_cols)}"

        df = df[["oare_id", "transliteration", "translation"]].copy()
        df["transliteration"] = df["transliteration"].fillna("").astype(str)
        df["translation"] = df["translation"].fillna("").astype(str)

        df.attrs["train_source"] = curated_path
        return df

    df = pd.read_csv(f"{INPUT_DIR}/train.csv")
    df.attrs["train_source"] = f"{INPUT_DIR}/train.csv"
    return df


train_df = load_train_df()
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")

In [6]:
train_source = train_df.attrs.get("train_source", "(unknown)")
print(f"Train Data: {len(train_df)} docs | source={train_source}")

Train Data: 1564 docs | source=/kaggle/input/datasets/tatsuokoshida/dpc-curated-train/train.curated.v002.xlsx


In [7]:
def simple_sentence_aligner(df):
    """
    【戦略の肝】
    Trainデータの「文書(複数文)」を、Testデータと同じ「文(1文)」に分割します。
    ここでは「英語の文数」と「アッカド語の行数」が一致する場合のみ分割する
    というヒューリスティック（簡易ルール）を使います。

    NOTE:
      - Sentence splitting increases samples per original document.
      - To avoid leakage in CV, we keep `oare_id` as a grouping key.
    """
    aligned_data = []

    for _, row in df.iterrows():
        oare_id = row["oare_id"]
        src = str(row["transliteration"])
        tgt = str(row["translation"])

        # Split the English text by sentence-ending punctuation.
        tgt_sents = [t.strip() for t in re.split(r"(?<=[.!?])\s+", tgt) if t.strip()]

        # Assume the Akkadian text is often separated by newlines and split accordingly.
        src_lines = [s.strip() for s in src.split('\n') if s.strip()]

        # If the counts match, trust it as 1-to-1 pairs and use the split version.
        if len(tgt_sents) > 1 and len(tgt_sents) == len(src_lines):
            for s, t in zip(src_lines, tgt_sents):
                if len(s) > 3 and len(t) > 3:  # Remove junk/noisy data.
                    aligned_data.append({"oare_id": oare_id, "transliteration": s, "translation": t})
        else:
            # If splitting fails (counts don't match), keep the original document pair as-is (safe fallback).
            aligned_data.append({"oare_id": oare_id, "transliteration": src, "translation": tgt})

    return pd.DataFrame(aligned_data, columns=["oare_id", "transliteration", "translation"])


In [8]:
# Run data augmentation.
train_expanded = simple_sentence_aligner(train_df)
print(f"Expanded Train Data: {len(train_expanded)} sentences (Alignment applied)")

train_expanded = train_expanded.reset_index(drop=True)
assert set(["oare_id", "transliteration", "translation"]).issubset(train_expanded.columns)

# ------------------------------------------
# Prefix multi-task: akk→en + en→akk (2x)
# ------------------------------------------
multitask_df = pd.concat(
    [
        train_expanded.assign(
            task="akk2en",
            source=train_expanded["transliteration"],
            target=train_expanded["translation"],
        ),
        train_expanded.assign(
            task="en2akk",
            source=train_expanded["translation"],
            target=train_expanded["transliteration"],
        ),
    ],
    ignore_index=True,
)[["oare_id", "task", "source", "target"]]

print(f"Multi-task Train Data: {len(multitask_df)} samples (2x directions)")
# Deterministic shuffle so batches mix directions
multitask_df = multitask_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(multitask_df.head())


Expanded Train Data: 1564 sentences (Alignment applied)
Multi-task Train Data: 3128 samples (2x directions)
                                oare_id    task  \
0  1f922cf5-3381-4e27-8473-d3ae9ce4e85a  en2akk   
1  093e14c7-c6f0-4ab1-a493-f6ed864ed0fa  en2akk   
2  477ce87f-8585-4113-bf83-611ddf62de8a  akk2en   
3  bcad41ab-0c33-43f2-b9ff-d2b30699f181  akk2en   
4  fe0b8448-f09e-47cf-b1e8-8e8d96531bb3  en2akk   

                                              source  \
0  To Aššur-muttabbil and Aššur-malik; to Aššur-m...   
1  From Aššuriš-tikal to Ali-ahum: Of the 9 talen...   
2  1.66666 ma-na KÙ.BABBAR ku-nu-ki-a a-na aḫ-ša-...   
3  0.33333 ma-na <gap> KÙ.BABBAR ṣa-ru-pu-um KI a...   
4  To Adida, Ali-ahum, and Uku from Puzur-Aššur: ...   

                                              target  
0  a-na a-šur-mu-ta-bi4-il5 ù a-šur-ma-lik a-na a...  
1  um-ma a-šur-iš-tí-kál-ma a-na a-lá-ḫi-im qí-bi...  
2  1.6666 mina of silver under my seal for Ah-šal...  
3  1/3 mina <gap> of refined

In [9]:
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

PREFIX_AKK_EN = "translate Akkadian to English: "
PREFIX_EN_AKK = "translate English to Akkadian: "

_SUBSCRIPT_NUM_MAP = str.maketrans({
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
})

_TRANSLATION_D1_PATTERNS = [
    (re.compile(r"\b(?:fem|sing|pl|plural)\.?(?=\s|$)", flags=re.IGNORECASE), " "),
    (re.compile(r"\(\?\)"), " "),
]


_GAP_MARKER_PATTERNS = [
    # Transliteration 側の gap 表現を `<gap>` に寄せる（host 推奨の test 整合観点）
    (re.compile(r"\(\s*(?:large\s+)?break\s*\)", flags=re.IGNORECASE), " <gap> "),
    (re.compile(r"\(\s*\d+\s+broken\s+lines?\s*\)", flags=re.IGNORECASE), " <gap> "),
    (re.compile(r"<\s*big_gap\s*>", flags=re.IGNORECASE), " <gap> "),
    (re.compile(r"\bbig_gap\b", flags=re.IGNORECASE), " <gap> "),
    (re.compile(r"\[\s*[xX]+\s*\]"), " <gap> "),
    (re.compile(r"\b[xX]{1,}\b"), " <gap> "),
]


def _collapse_gaps(text: str) -> str:
    # `<gap>` の前後の空白を一旦整えてから縮約する
    text = re.sub(r"\s*<gap>\s*", " <gap> ", text)

    # `<gap>. <gap>` / `<gap>, <gap>` のように句読点を挟んだ連続も `<gap>` に縮約
    while True:
        new_text = re.sub(r"<gap>\s*[\.,]\s*<gap>", "<gap>", text)
        new_text = re.sub(r"<gap>\s+<gap>", "<gap>", new_text)
        if new_text == text:
            break
        text = new_text
    return text


def normalize_transliteration(text):
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(_SUBSCRIPT_NUM_MAP)

    # ダッシュ類の統一
    text = re.sub(r"[‐‑–—]", "-", text)

    # 下付き `ₓ` の除去（ノイズになりやすい）
    text = text.replace("ₓ", "")

    # 長い小数を短縮（小数点以下4桁まで）
    text = re.sub(r"(\d+\.\d{4})\d+", r"\1", text)

    # gap 表現を `<gap>` に統一
    text = text.replace("…", " <gap> ")
    for pat, repl in _GAP_MARKER_PATTERNS:
        text = pat.sub(repl, text)

    text = _collapse_gaps(text)

    # Collapse whitespace noise introduced by normalization/replacements.
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_translation_d1(text):
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)

    for pattern, repl in _TRANSLATION_D1_PATTERNS:
        text = pattern.sub(repl, text)

    # `PN` は `<gap>` に寄せる（host 推奨）
    text = re.sub(r"\bPN\b", " <gap> ", text)

    text = _collapse_gaps(text)

    text = re.sub(r"\s+", " ", text).strip()
    return text


def postprocess_translation(text):
    # 生成後の後処理用（推論Notebook側で同等の関数を使う想定）
    return normalize_translation_d1(text)


def preprocess_function(examples):
    tasks = examples["task"]
    sources = examples["source"]
    targets = examples["target"]

    inputs = []
    out_targets = []

    for t, s, y in zip(tasks, sources, targets):
        if t == "akk2en":
            s_norm = normalize_transliteration(s)
            y_norm = normalize_translation_d1(y)
            inputs.append(PREFIX_AKK_EN + s_norm)
            out_targets.append(y_norm)
        elif t == "en2akk":
            inputs.append(PREFIX_EN_AKK + str(s))
            out_targets.append(normalize_transliteration(y))
        else:
            raise ValueError(f"Unknown task: {t}")

    model_inputs = tokenizer(inputs, max_length=Config.MAX_LENGTH, truncation=True)
    labels = tokenizer(out_targets, max_length=Config.MAX_LENGTH, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs



tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [10]:
# ==========================================
# 4. Full-data training (fine-tuning)
# ==========================================

# Build a single HF dataset (full train, no CV).
ds_train = Dataset.from_pandas(multitask_df)

# Tokenize
train_tokenized = ds_train.map(
    preprocess_function,
    batched=True,
    remove_columns=ds_train.column_names,
)

# Model
model = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME)
model.config.use_cache = False
model.gradient_checkpointing_enable()
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

args = Seq2SeqTrainingArguments(
    output_dir=Config.OUTPUT_DIR,

    # No CV / no validation in this notebook
    eval_strategy="no",

    save_strategy="no",

    # Checkpoints (incl. optimizer states) can be huge; save only final model at the end.

    learning_rate=Config.LEARNING_RATE,
    fp16=False,

    # Memory savers
    gradient_checkpointing=True,
    group_by_length=True,

    per_device_train_batch_size=Config.PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS,

    weight_decay=0.01,
    num_train_epochs=Config.EPOCHS,

    predict_with_generate=False,

    logging_strategy="steps",
    logging_steps=200,
    disable_tqdm=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting Training (full train, multi-task, FP32 mode)...")
trainer.train()

trainer.save_model(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)
print(f"Saved model to: {Config.OUTPUT_DIR}")


Map:   0%|          | 0/3128 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/tmp/ipykernel_24/2746493822.py:54: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting Training (full train, multi-task, FP32 mode)...
{'loss': 1.2358, 'grad_norm': 1.797877311706543, 'learning_rate': 0.00019491048593350385, 'epoch': 0.5115089514066496}
{'loss': 0.824, 'grad_norm': 0.4696495831012726, 'learning_rate': 0.00018979539641943735, 'epoch': 1.0230179028132993}
{'loss': 0.7077, 'grad_norm': 0.5821518301963806, 'learning_rate': 0.00018468030690537085, 'epoch': 1.5345268542199488}
{'loss': 0.6674, 'grad_norm': 0.36499297618865967, 'learning_rate': 0.00017956521739130436, 'epoch': 2.0460358056265986}
{'loss': 0.5783, 'grad_norm': 0.33598992228507996, 'learning_rate': 0.00017445012787723786, 'epoch': 2.557544757033248}
{'loss': 0.5595, 'grad_norm': 0.4105969965457916, 'learning_rate': 0.00016933503836317137, 'epoch': 3.0690537084398977}
{'loss': 0.4997, 'grad_norm': 0.403167188167572, 'learning_rate': 0.00016421994884910487, 'epoch': 3.580562659846547}
{'loss': 0.4634, 'grad_norm': 0.6034296751022339, 'learning_rate': 0.00015910485933503838, 'epoch': 4.0920

In [11]:
# --- Notes ---
# This notebook trains a single model using all training data (no CV).
# Train:
#   - curated xlsx を使う場合は USE_CURATED_TRAIN=True のまま、
#     CURATED_TRAIN_XLSX_CANDIDATES のパスを Kaggle / ローカル実行環境に合わせて調整してください。
# Output:
#   ./byt5-akkadian-model/fulltrain_byt5-base_multitask/
